# ARC3 Colab hardware check
Runtime > Change runtime type > **G4 (RTX PRO 6000)**, then run the cell below and paste its output back.

In [ ]:
# ARC3 on Colab: can this runtime run the Kaggle RTX PRO 6000 workload? (~1 min, no downloads)
# Needs: RTX PRO 6000 (96 GB), NVIDIA driver >= 580 (CUDA 13 wheels), host RAM >= 170 GB
# (102 GB PLE table is pinned in host RAM), free disk >= 220 GB (model files ~210 GB), Python 3.12.
import os, re, shutil, subprocess, sys

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

checks = []
gpu = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
print("GPU      :", gpu or "none")
name, mem, drv = ([x.strip() for x in gpu.split(",")] + ["", "", ""])[:3] if gpu else ("", "", "0")
checks.append(("GPU is RTX PRO 6000", "RTX PRO 6000" in name.upper()))
checks.append(("driver >= 580", int((drv.split(".") or ["0"])[0] or 0) >= 580))

try:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
except (AttributeError, ValueError, OSError):
    ram_gb = 0.0
print(f"host RAM : {ram_gb:.1f} GB")
checks.append(("host RAM >= 170 GB", ram_gb >= 170))

best = max(("/", "/content", "/tmp", "/mnt"), key=lambda p: shutil.disk_usage(p).free if os.path.exists(p) else 0)
free_gb = shutil.disk_usage(best).free / 1e9
print(sh("df -h / /content /tmp /mnt 2>/dev/null"))
print(f"most free: {best} {free_gb:.0f} GB")
checks.append(("free disk >= 220 GB", free_gb >= 220))

print("python   :", sys.version.split()[0], "| CPUs:", os.cpu_count(), "| memlock:", sh("ulimit -l"), "| user:", sh("id -un"))
checks.append(("python 3.12", sys.version_info[:2] == (3, 12)))
print(sh("nvidia-smi | head -4 | tail -2"))

print()
for what, ok in checks:
    print(("PASS " if ok else "FAIL ") + what)
print("\nRESULT:", "ALL PASS - this runtime can run the ARC3 server" if all(ok for _, ok in checks)
      else "NOT ENOUGH - paste this output back to Claude")
